In [ ]:
from pathlib import Path
import hashlib
import json
import re
import pandas as pd
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
SEED = CONFIG['seed']
OUTPUT.mkdir(exist_ok=True)
DECIMALS = 3
PREFIXES = {'empirical_complete':'DBLP', 'arw4':'ARWFour', 'lognormal_grw':'GRW', 'ar1_s':'AROne',
            'hurdle_ar1_s_p':'HurdleAROne', 'ar6_6':'ARSix', 'se_hurdle_s':'SEHurdle'}
WINDOWS = {'last_four':'FourYear', 'career_total':'Career'}
TOTAL_METRICS = [('QQRtwo','qq_r2',DECIMALS,1), ('MeanTotal','mean_total',DECIMALS,1),
                 ('Gini','gini_total',DECIMALS,1), ('FitKSD','ks_D',DECIMALS,1),
                 ('NTotal','n_total',0,1), ('NPositive','n_positive',0,1),
                 ('NZero','n_zero',0,1), ('ZeroPercent','zero_fraction',1,100)]
MOMENT_METRICS = [('MedianTotal','median'), ('SDTotal','sd'), ('VarianceTotal','variance')]
RANK_RMSE_METRICS = [('RankInitialRMSE','age_0_rmse11_20'), ('RankLagThreeRMSE','lag_3_rmse11_20'),
                     ('RankLagFiveRMSE','lag_5_rmse11_20')]
RANK_METRICS = [('RankInitialFinal','age_0_rho20'), ('RankLagThreeFinal','lag_3_rho20'),
                ('RankLagFiveFinal','lag_5_rho20')]
FILES = {'lognormality':INPUT/'diagnostics/lognormality_summary.csv',
         'moments':INPUT/'diagnostics/historic_distribution_summary.csv',
         'comparison':INPUT/'diagnostics/model_comparison.csv',
         'ranks':INPUT/'diagnostics/rank_correlations.csv', 'config':ROOT/'config.json'}

In [ ]:
totals = pd.read_csv(FILES['lognormality']).set_index(['source','window'])
moments = pd.read_csv(FILES['moments']).set_index(['source','diagnostic'])
comparison = pd.read_csv(FILES['comparison']).set_index('source')
# last four endpoint years for fixed-point and five-year-lag annual ranks
rank_start, rank_end = totals.loc[('empirical_complete','last_four'), ['start_age','end_age']]
ranks = pd.read_csv(FILES['ranks'])
rank_references = {'age_0':'RankInitialFourYearSlope', 'lag_5':'RankLagFiveFourYearSlope'}
ranks = ranks.loc[ranks.measure.eq('annual') & ranks.age.between(rank_start,rank_end)
                  & ranks.reference.isin(rank_references)]
ages = ranks.age - (rank_start+rank_end)/2
# OLS slope with an intercept, on the common four-year grid
slopes = ranks.assign(weighted_rho=ages*ranks.rho, age_sq=ages**2).groupby(['source','reference'])
slopes = slopes.weighted_rho.sum()/slopes.age_sq.sum()

In [ ]:
rows = []
def add_macro(name, value, table, source, window, statistic, decimals=DECIMALS, scale=1):
    if name.endswith(('MeanTotal', 'MedianTotal')):
        decimals = 0
    elif name.endswith('SDTotal'):
        decimals = 1
    elif name.endswith('Gini'):
        decimals = 3
    grouping = ',' if statistic in ('n_total', 'n_positive', 'mean_total', 'median', 'sd') else ''
    rows.append({'macro':name, 'table':table, 'source':source, 'window':window, 'statistic':statistic,
                 'raw_value':value, 'scale':scale, 'decimals':decimals,
                 'formatted_value':f'{value*scale:{grouping}.{decimals}f}', 'alias_of':''})

empirical = totals.loc[('empirical_complete','last_four')]
for name,column in [('DBLPNComplete','n_total'), ('DBLPFourYearStart','start_age'), ('DBLPFourYearEnd','end_age')]:
    add_macro(name,empirical[column],'lognormality','empirical_complete','last_four',column,0)
add_macro('ModelSeed',SEED,'config','','','seed',0)
for source,prefix in PREFIXES.items():
    for window,window_label in WINDOWS.items():
        selected = totals.loc[(source,window)]
        for suffix,column,decimals,scale in TOTAL_METRICS:
            add_macro(prefix+window_label+suffix,selected[column],'lognormality',source,window,column,decimals,scale)
        for suffix,column in MOMENT_METRICS:
            add_macro(prefix+window_label+suffix,moments.loc[(source,window),column],'moments',source,window,column)
    for reference,suffix in rank_references.items():
        add_macro(prefix+suffix,slopes.loc[(source,reference)],'ranks',source,
                  f'annual_{int(rank_start)}_{int(rank_end)}',reference,4)
    selected = comparison.loc[source]
    if source != 'empirical_complete':
        for suffix,column in RANK_RMSE_METRICS:
            add_macro(prefix+suffix,selected[column],'comparison',source,'annual_11_20',column,4)
    for suffix,column in RANK_METRICS:
        add_macro(prefix+suffix,selected[column],'comparison',source,'annual_final',column)
    add_macro(prefix+'FinalYearZeroPercent',selected.final_zero_fraction,'comparison',source,'annual_final',
              'final_zero_fraction',1,100)
catalog = pd.DataFrame(rows)

In [5]:
# short QQ names always refer to the primary four-year window
aliases = {prefix+'QQRtwo':prefix+'FourYearQQRtwo' for prefix in PREFIXES.values()}
aliases['GRWLognormalRtwo'] = 'GRWFourYearQQRtwo'
alias_rows = []
for name,target in aliases.items():
    row = catalog.set_index('macro').loc[target].to_dict()
    alias_rows.append({'macro':name,**row,'alias_of':target})
catalog = pd.concat([catalog,pd.DataFrame(alias_rows)],ignore_index=True)

In [ ]:
# only the model comparison table defines primary result tags
paper = (ROOT.parent/'manuscript/paper/report_main.tex').read_text()
tables = re.findall(r'\\begin\{table\*?\}.*?\\end\{table\*?\}', paper, flags=re.S)
comparison_table = next((table for table in tables if r'\label{tab:model-comparison}' in table), '')
primary_macros = set(re.findall(r'\\([A-Za-z]+)', comparison_table))
catalog['tag'] = catalog.macro.map(lambda name: name if name in primary_macros else 'si_'+name)
# record both the input link and its authoritative CSV; no researcher-level data
provenance = pd.DataFrame([{'table':key, 'input_path':str(path.relative_to(ROOT.parent)),
                            'source_file':str(path.resolve().relative_to(ROOT.parent)),
                            'sha256':hashlib.sha256(path.read_bytes()).hexdigest()} for key,path in FILES.items()])
manifest = catalog.merge(provenance,on='table')
manifest.to_csv(OUTPUT/'macro_manifest.csv',index=False)
provenance.to_csv(OUTPUT/'report_sources.csv',index=False)
header = [r'% generated by dblp_models/g_report/src/report.ipynb; refresh with make report',
          r'% FourYear = summed final four years; Career = whole-career supplement',
          r'% QQ and FitKSD use positive totals; raw moments and Gini include zeros',
          r'% Rank RMSE uses annual rank curves over endpoint years 11--20',
          r'% Percent values omit the percent sign']
definitions = []

for row in catalog.itertuples():
    value = '\\'+row.alias_of if row.alias_of else row.formatted_value
    if row.tag != row.macro:
        definitions.append(r'\expandafter\newcommand\csname '+row.tag+r'\endcsname{'+value+r'\xspace}')
        value = r'\csname '+row.tag+r'\endcsname'
    definitions.append('\\newcommand{\\'+row.macro+'}{'+value+'\\xspace}')

# atomic replacement while an editor may be compiling the manuscript
temporary = OUTPUT/'results.tex.tmp'
temporary.write_text('\n'.join(header+definitions)+'\n')
temporary.replace(OUTPUT/'results.tex')
display(catalog.loc[catalog.macro.isin(['DBLPNComplete','DBLPQQRtwo','GRWQQRtwo','SEHurdleQQRtwo',
                                      'DBLPRankInitialFinal','DBLPRankLagThreeFinal','DBLPRankLagFiveFinal']),
                    ['macro','formatted_value','source','window','statistic']])